[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MrEGAMERZ/netguard-ai/blob/main/NetGuard_AI_Intrusion_Detection.ipynb)

# NetGuard AI: Machine Learning Based Network Intrusion Detection System

This notebook trains two machine learning models on network traffic data to detect whether a connection is benign (normal traffic) or an attack (malicious activity).

The dataset used is the Canadian Institute for Cybersecurity Intrusion Detection System 2017 dataset, commonly known as CIC-IDS2017. It contains labeled network flow records that cover several categories of cyber threats, including Denial of Service attacks, brute force password guessing, port scanning, botnets, and web application attacks.

### What this project does
* Loads two hundred thousand recorded network flow records from the CIC-IDS2017 dataset
* Cleans the data by fixing infinite values and missing numbers
* Groups all specific attack categories into one general Attack label for clear binary classification
* Trains a Random Forest classifier as our baseline model on the Central Processing Unit
* Trains a Multi Layer Perceptron neural network using Graphics Processing Unit acceleration
* Evaluates both models using Accuracy, Precision, Recall, and the F1 Score
* Demonstrates an interactive prediction function where you can test individual traffic records

### What this project does not do
* It does not monitor live Wi-Fi or inspect packets in real time
* It does not block or stop attacks on your computer
* It does not read raw packet payloads (it uses statistical flow summaries)

### Runtime setup
For the best speed, go to the top menu in Google Colab:
Click **Runtime** -> **Change runtime type** -> select **T4 GPU** -> click **Save**.
The neural network trains faster on a Graphics Processing Unit, although it will also work on a standard Central Processing Unit.

## Stage 1: Install Dependencies

The `datasets` library from Hugging Face lets us stream large datasets without downloading them in full upfront.


In [ ]:
# Install the Hugging Face datasets library
!pip install -q datasets
print("Installation complete.")


## Stage 2: Import Libraries

We import everything needed for data loading, preprocessing, model training, and evaluation.

| Library | Purpose |
|---|---|
| NumPy | Numerical computing and array operations |
| Pandas | Loading and manipulating tabular data |
| Matplotlib / Seaborn | Plotting graphs and charts |
| datasets | Streaming the CIC-IDS2017 dataset from Hugging Face |
| scikit-learn | Preprocessing, Random Forest, and evaluation metrics |
| TensorFlow / Keras | Building and training the neural network |

`RANDOM_STATE = 42` is a fixed seed. It makes randomized steps like data shuffling and weight initialization produce the same result every time you run the notebook. It does not affect model accuracy.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("All libraries loaded.")
print(f"TensorFlow version: {tf.__version__}")
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print(f"Running on: {device}")


## Stage 3: Load the Dataset

We download a sample of 200,000 records from the CIC-IDS2017 dataset hosted on Hugging Face.

The full dataset is several gigabytes. A 200,000-record sample keeps training time reasonable in Colab while still giving the models enough data to learn meaningful patterns.

Streaming mode reads records progressively instead of downloading the entire dataset into memory first. The `shuffle` step mixes the record order using a bounded in-memory buffer before we take our sample.


In [ ]:
print("Connecting to Hugging Face and loading dataset...")
print("This usually takes 2 to 4 minutes depending on your connection speed.")
print()

dataset = load_dataset(
    "muzom/CIC-IDS2017",
    split="train",
    streaming=True,
)

sample = dataset.shuffle(
    seed=RANDOM_STATE,
    buffer_size=50_000,
).take(200_000)

df = pd.DataFrame(list(sample))

# Strip any leading or trailing spaces from column names
df.columns = df.columns.str.strip()

print(f"Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head(3)


## Stage 4: Inspect the Data

Before writing any preprocessing code, we need to understand what the data looks like. This includes checking column names, data types, missing values, and the distribution of traffic labels.

A single row in this dataset represents one network flow. A network flow is a sequence of packets between two endpoints sharing the same source address, destination address, source port, destination port, and protocol. Each column represents a statistical property of that flow, such as its duration, the number of packets sent in each direction, or the average packet size.


In [ ]:
print("Column names:")
print(df.columns.tolist())


In [ ]:
print("Data types and non-null counts per column:")
df.info()


In [ ]:
print("Missing values per column (showing top 15 with most missing):")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].head(15))
print()
print("Original traffic label distribution:")
print(df["Label"].value_counts())


In [ ]:
# Visualize the original label distribution
label_counts = df["Label"].value_counts()

plt.figure(figsize=(13, 5))
bars = plt.bar(
    label_counts.index,
    label_counts.values,
    color="#4a90d9",
    edgecolor="black",
    linewidth=0.4,
)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.ylabel("Number of Records")
plt.title("Traffic Label Distribution in the 200,000-Record Sample")
plt.tight_layout()
plt.show()

print(f"Total unique labels: {df['Label'].nunique()}")
print(f"Total records: {len(df):,}")


## Stage 5: Clean and Preprocess the Data

The raw dataset needs several preparation steps before training.

**Label conversion.** CIC-IDS2017 contains many specific attack categories such as "DoS Hulk", "PortScan", and "Bot". We convert all of these into a single Attack label. The result is a binary classification problem: each record is either Benign or Attack. This keeps the project manageable and makes the output easy to explain.

**Infinite values.** Some computed features like "Flow Bytes per Second" can produce infinite values when the flow duration is zero. We replace these with the median value of that column.

**Missing values.** Any remaining missing values after the infinite-value replacement are filled with the column median.

**Feature selection.** We keep only numeric columns. Non-numeric columns such as IP addresses and timestamps are identifiers, not predictive features for a general classifier.


In [ ]:
# Convert multi-class labels to binary: Benign = 0, Attack = 1
df["binary_label"] = df["Label"].apply(
    lambda x: 0 if str(x).strip().upper() == "BENIGN" else 1
)

benign_count = (df["binary_label"] == 0).sum()
attack_count = (df["binary_label"] == 1).sum()
total = len(df)

print("Binary label distribution:")
print(f"  Benign: {benign_count:,}  ({benign_count / total * 100:.1f}%)")
print(f"  Attack: {attack_count:,}  ({attack_count / total * 100:.1f}%)")
print()

# Visualize binary distribution
plt.figure(figsize=(5, 4))
plt.bar(["Benign", "Attack"], [benign_count, attack_count],
        color=["#2ca02c", "#d62728"], edgecolor="black", linewidth=0.5)
plt.ylabel("Number of Records")
plt.title("Binary Label Distribution")
for i, v in enumerate([benign_count, attack_count]):
    plt.text(i, v + 500, f"{v:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Select numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if "binary_label" in numeric_cols:
    numeric_cols.remove("binary_label")

print(f"Numeric feature columns found: {len(numeric_cols)}")

# Replace infinite values with NaN, then fill NaN with the column median
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Drop any columns that ended up entirely empty
valid_cols = [c for c in numeric_cols if df[c].isnull().sum() == 0]

print(f"Columns remaining after cleaning: {len(valid_cols)}")
print(f"Total missing values remaining: {df[valid_cols].isnull().sum().sum()}")


In [ ]:
# Select the most informative network flow features
# These capture volume, timing, size, and flag characteristics of each flow

preferred_features = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Packet Length Mean",
    "Packet Length Std",
    "Average Packet Size",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "ACK Flag Count",
    "SYN Flag Count",
    "PSH Flag Count",
    "RST Flag Count",
    "Active Mean",
    "Idle Mean",
]

selected_features = [f for f in preferred_features if f in valid_cols]

if len(selected_features) < 10:
    print("Fewer than 10 preferred features found. Using all numeric columns.")
    selected_features = valid_cols
else:
    print(f"Using {len(selected_features)} selected features:")
    for f in selected_features:
        print(f"  {f}")


## Stage 6: Prepare Training and Testing Data

We split the data into a training set (80%) and a testing set (20%). The model learns only from the training set. The testing set is kept completely separate to measure how well the model generalises to data it has never seen before.

After splitting, we apply Standard Scaling to normalize the feature values. Standard Scaling adjusts each feature to have a mean of zero and a standard deviation of one. This prevents features with large numeric ranges such as "Flow Bytes per Second" from drowning out features with small ranges such as flag counts.

The scaler is fitted on training data only. Fitting on the full dataset before splitting would be data leakage, which produces artificially inflated evaluation scores.


In [ ]:
X = df[selected_features].values
y = df["binary_label"].values

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape:  {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,  # Keep the same class ratio in both splits
)

print(f"\nTraining set:  {X_train.shape[0]:,} records")
print(f"Testing set:   {X_test.shape[0]:,} records")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\nStandard Scaling applied.")
print(f"Training feature mean (expected near 0): {X_train_scaled.mean():.4f}")
print(f"Training feature std  (expected near 1): {X_train_scaled.std():.4f}")


## Stage 7: Random Forest Classifier

A Random Forest is a collection of decision trees. Each tree is trained on a different random subset of the training data and makes its own prediction. The final output is the majority vote across all trees.

Random Forests are a strong baseline for tabular data. They handle non-linear relationships well, are resistant to overfitting, and naturally produce a measure of which features contributed most to the predictions.

We use 100 trees with a maximum depth of 20. The `n_jobs=-1` setting tells scikit-learn to use all available CPU cores in parallel, which speeds up training.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest classifier on CPU...")
print("Expected time: 1 to 3 minutes depending on your Colab instance.")
print()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

rf_model.fit(X_train_scaled, y_train)
print("\nRandom Forest training complete.")


In [ ]:
# Evaluate Random Forest on the test set
rf_predictions = rf_model.predict(X_test_scaled)

rf_accuracy   = accuracy_score(y_test, rf_predictions)
rf_precision  = precision_score(y_test, rf_predictions, zero_division=0)
rf_recall     = recall_score(y_test, rf_predictions, zero_division=0)
rf_f1         = f1_score(y_test, rf_predictions, zero_division=0)

print("Random Forest Results on the Test Set")
print("=" * 42)
print(f"Accuracy:   {rf_accuracy   * 100:.2f}%")
print(f"Precision:  {rf_precision  * 100:.2f}%")
print(f"Recall:     {rf_recall     * 100:.2f}%")
print(f"F1 Score:   {rf_f1         * 100:.2f}%")
print()
print("Full Classification Report:")
print(classification_report(y_test, rf_predictions, target_names=["Benign", "Attack"]))


In [ ]:
# Confusion matrix for Random Forest
cm_rf = confusion_matrix(y_test, rf_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_rf, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Benign", "Attack"],
    yticklabels=["Benign", "Attack"],
)
plt.title("Random Forest: Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm_rf.ravel()
print(f"True Negatives  (Benign correctly identified as Benign): {tn:,}")
print(f"True Positives  (Attack correctly identified as Attack): {tp:,}")
print(f"False Positives (Benign incorrectly flagged as Attack):  {fp:,}")
print(f"False Negatives (Attack that slipped through as Benign): {fn:,}")


In [ ]:
# Feature importance: which features the Random Forest relied on most
importances = rf_model.feature_importances_
top_n = min(15, len(selected_features))
indices = np.argsort(importances)[::-1][:top_n]

plt.figure(figsize=(11, 6))
plt.bar(
    range(top_n),
    importances[indices],
    color="#4a90d9",
    edgecolor="black",
    linewidth=0.4,
)
plt.xticks(
    range(top_n),
    [selected_features[i] for i in indices],
    rotation=45,
    ha="right",
    fontsize=9,
)
plt.ylabel("Importance Score")
plt.title(f"Random Forest: Top {top_n} Most Important Features")
plt.tight_layout()
plt.show()


## Stage 8: Neural Network (Multi-Layer Perceptron)

A Multi-Layer Perceptron (MLP) is a feedforward neural network. Data flows in one direction: from the input layer, through hidden layers, to the output layer.

Each hidden layer contains neurons that compute a weighted sum of their inputs and pass the result through an activation function. The Rectified Linear Unit (ReLU) activation function is used here because it allows the network to learn non-linear relationships without suffering from the vanishing gradient problem that affects older activation functions.

**Network architecture:**
- Input layer: one node per feature
- Hidden layer 1: 128 neurons, ReLU activation, Batch Normalization, 30% Dropout
- Hidden layer 2: 64 neurons, ReLU activation, Batch Normalization, 20% Dropout
- Hidden layer 3: 32 neurons, ReLU activation
- Output layer: 1 neuron, Sigmoid activation

**Batch Normalization** normalizes the activations within each mini-batch. This stabilizes training and allows the use of higher learning rates.

**Dropout** randomly disables a percentage of neurons during each training step. This forces the network to learn redundant representations and reduces overfitting.

**Sigmoid** output produces a value between 0 and 1. We interpret values above 0.5 as Attack and below 0.5 as Benign.

Training uses the **Adam optimizer** (Adaptive Moment Estimation) with **binary cross-entropy loss**, which are standard choices for binary classification.


In [ ]:
n_features = X_train_scaled.shape[1]

nn_model = keras.Sequential([
    layers.Input(shape=(n_features,)),

    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.30),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.20),

    layers.Dense(32, activation="relu"),

    layers.Dense(1, activation="sigmoid"),
], name="NetGuard_MLP")

nn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

nn_model.summary()


In [ ]:
print("Training neural network...")
print()

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,  # Revert to the epoch with the lowest validation loss
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,     # Halve the learning rate when progress stalls
    patience=2,
    min_lr=1e-6,
    verbose=1,
)

history = nn_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=30,
    batch_size=512,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

print(f"\nTraining stopped at epoch {len(history.history['loss'])}.")


In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["accuracy"],     label="Training Accuracy",   color="#2ca02c")
ax1.plot(history.history["val_accuracy"], label="Validation Accuracy", color="#d62728", linestyle="--")
ax1.set_title("Neural Network: Accuracy over Epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history.history["loss"],     label="Training Loss",   color="#1f77b4")
ax2.plot(history.history["val_loss"], label="Validation Loss", color="#ff7f0e", linestyle="--")
ax2.set_title("Neural Network: Loss over Epochs")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Binary Cross-Entropy Loss")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate Neural Network on the test set
nn_probs      = nn_model.predict(X_test_scaled, verbose=0).flatten()
nn_predictions = (nn_probs >= 0.5).astype(int)

nn_accuracy   = accuracy_score(y_test, nn_predictions)
nn_precision  = precision_score(y_test, nn_predictions, zero_division=0)
nn_recall     = recall_score(y_test, nn_predictions, zero_division=0)
nn_f1         = f1_score(y_test, nn_predictions, zero_division=0)

print("Neural Network Results on the Test Set")
print("=" * 42)
print(f"Accuracy:   {nn_accuracy   * 100:.2f}%")
print(f"Precision:  {nn_precision  * 100:.2f}%")
print(f"Recall:     {nn_recall     * 100:.2f}%")
print(f"F1 Score:   {nn_f1         * 100:.2f}%")
print()
print("Full Classification Report:")
print(classification_report(y_test, nn_predictions, target_names=["Benign", "Attack"]))

cm_nn = confusion_matrix(y_test, nn_predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_nn, annot=True, fmt="d", cmap="Oranges",
    xticklabels=["Benign", "Attack"],
    yticklabels=["Benign", "Attack"],
)
plt.title("Neural Network: Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()


## Stage 9: Model Comparison

We compare both models on the same held-out test set across four metrics.

**Accuracy** is the percentage of all records classified correctly. It is the most intuitive metric but can mislead when one class is much more common than the other.

**Precision** answers: of all the connections we flagged as attacks, how many actually were attacks? Low precision means too many false alarms, which wastes analyst time.

**Recall** answers: of all the actual attacks in the test set, how many did we detect? Low recall means attacks are slipping through unnoticed.

**F1 Score** is the harmonic mean of precision and recall. It gives a single number that penalizes models that are very strong on one metric but weak on the other.

In a real intrusion detection system, recall is typically the more important metric. A missed attack is usually more damaging than a false alarm.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Neural Network (MLP)"],
    "Accuracy (%)":  [round(rf_accuracy  * 100, 2), round(nn_accuracy  * 100, 2)],
    "Precision (%)": [round(rf_precision * 100, 2), round(nn_precision * 100, 2)],
    "Recall (%)":    [round(rf_recall    * 100, 2), round(nn_recall    * 100, 2)],
    "F1 Score (%)":  [round(rf_f1        * 100, 2), round(nn_f1        * 100, 2)],
}).set_index("Model")

print("Model Comparison on Held-Out Test Set")
print("=" * 55)
print(comparison.to_string())
print()

best_f1 = comparison["F1 Score (%)"].idxmax()
print(f"Higher F1 Score: {best_f1}")


In [ ]:
# Side-by-side bar chart
metrics = ["Accuracy (%)", "Precision (%)", "Recall (%)", "F1 Score (%)"]
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))

bars1 = ax.bar(
    x - width / 2,
    comparison.loc["Random Forest", metrics],
    width, label="Random Forest",
    color="#4a90d9", edgecolor="black", linewidth=0.4,
)
bars2 = ax.bar(
    x + width / 2,
    comparison.loc["Neural Network (MLP)", metrics],
    width, label="Neural Network (MLP)",
    color="#e67e22", edgecolor="black", linewidth=0.4,
)

ax.set_xlabel("Metric")
ax.set_ylabel("Score (%)")
ax.set_title("Random Forest vs Neural Network: Performance Comparison")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(80, 101)
ax.legend()
ax.bar_label(bars1, fmt="%.1f%%", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.1f%%", padding=3, fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Stage 10: Prediction Demo

The function below takes a single network flow record and predicts whether the traffic is benign or an attack. It shows the prediction and confidence score from both models.

You can call it with a specific index from the test set, or let it pick a random record.


In [ ]:
def predict_traffic(record_index=None):
    """
    Predict the class of a single network flow record.

    Parameters
    ----------
    record_index : int, optional
        Index into the test set. A random index is chosen if not provided.
    """
    if record_index is None:
        record_index = np.random.randint(0, len(X_test))

    record = X_test_scaled[record_index].reshape(1, -1)
    actual = "Attack" if y_test[record_index] == 1 else "Benign"

    rf_pred   = rf_model.predict(record)[0]
    rf_conf   = rf_model.predict_proba(record)[0][rf_pred]
    nn_prob   = float(nn_model.predict(record, verbose=0)[0][0])
    nn_pred   = int(nn_prob >= 0.5)

    rf_label  = "Attack" if rf_pred == 1 else "Benign"
    nn_label  = "Attack" if nn_pred == 1 else "Benign"

    print(f"Test record index:          {record_index}")
    print(f"Actual label:               {actual}")
    print(f"Random Forest prediction:   {rf_label}  (confidence: {rf_conf * 100:.1f}%)")
    print(f"Neural Network prediction:  {nn_label}  (attack probability: {nn_prob * 100:.1f}%)")

    if rf_label == nn_label == actual:
        print("Both models correct.")
    elif rf_label == nn_label:
        print("Both models agree but prediction is wrong.")
    else:
        print("Models disagree.")
    print()


In [ ]:
# Run the demo on 8 randomly chosen records from the test set
print("Prediction demo on 8 randomly selected test records")
print("=" * 50)
for _ in range(8):
    predict_traffic()


## Summary

This notebook demonstrated a complete machine learning pipeline for binary network intrusion detection.

**What was built:**
1. A data loading and preprocessing pipeline handling infinite values, missing data, and multi-class label reduction
2. A Random Forest classifier trained on selected network flow features
3. A Multi-Layer Perceptron neural network with batch normalization, dropout, and early stopping
4. A side-by-side comparison using accuracy, precision, recall, and F1 score
5. An interactive prediction function

**Key things to understand and explain:**

The dataset was collected in 2017. Attack patterns evolve, and newer threats are not represented here. This model would need retraining on more recent data to remain useful over time.

We used a 200,000-record sample, not the full dataset. Results may differ slightly on the complete data.

High performance on this dataset does not guarantee the same results on live production traffic. CIC-IDS2017 attack traffic has clear statistical signatures. Real-world attacks can be deliberately crafted to blend in with normal traffic.

**What to say if asked in a viva or oral examination:**

- "CIC-IDS2017 is a standard research dataset from the Canadian Institute for Cybersecurity. It contains labeled network flow records covering several attack types."
- "We converted 15 specific attack categories into a single binary label to keep the problem focused and the output easy to explain."
- "A network flow feature is a statistical summary of a connection, like the number of packets sent, the average packet size, or how long the connection lasted. The model never sees the actual packet contents."
- "Random Forest works by training many decision trees on different subsets of data. Each tree votes, and the majority vote wins."
- "The neural network uses backpropagation. After each batch of records, it computes how wrong the predictions were, calculates the gradient of the loss with respect to each weight, and adjusts the weights in the direction that reduces the loss."
- "Recall is the more important metric here because missing a real attack is typically more costly than raising a false alarm."
- "This system analyzes recorded network flow data. It does not monitor live traffic or block anything."
